In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from scipy.stats import pearsonr, spearmanr
from sklearn.preprocessing import MinMaxScaler



# 1 Define input and output files


BASE_DIR = Path.cwd()

INPUT_PATH = (
    BASE_DIR
    / "clear_test_readabert_gaze_rerun.csv"
)

OUTPUT_PATH = (
    BASE_DIR
    / "clear_test_readabert_gaze_cognitive_rerun.csv"
)

GAZE_LENGTH_PLOT = (
    BASE_DIR
    / "phase8_gaze_vs_length_rerun.png"
)

D_C_PLOT = (
    BASE_DIR
    / "phase8_difficulty_vs_cognitive_effort_rerun.png"
)

print(
    "Input file exists:",
    INPUT_PATH.exists()
)

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Phase 7 output not found: {INPUT_PATH}"
    )



# 2 Load the CLEAR data with gaze predictions


df = pd.read_csv(
    INPUT_PATH
)

required_columns = [
    "Excerpt",
    "Difficulty_D",
    "Gaze_Score_Raw",
    "token_count_gaze"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        "Missing required columns: "
        + ", ".join(
            missing_columns
        )
    )

print(
    "\nShape:",
    df.shape
)

print(
    "\nInput sample:"
)

print(
    df[
        [
            "Excerpt",
            "Difficulty_D",
            "Gaze_Score_Raw",
            "token_count_gaze"
        ]
    ].head()
)


# 3 Inspect the raw gaze-score distribution


print(
    "\nRaw gaze-score distribution:"
)

print(
    df[
        "Gaze_Score_Raw"
    ].describe()
)

print(
    "\nMissing gaze scores:",
    df[
        "Gaze_Score_Raw"
    ]
    .isna()
    .sum()
)



# 4 Inspect gaze-token counts


print(
    "\nGaze-token count distribution:"
)

print(
    df[
        "token_count_gaze"
    ].describe()
)



# 5 Test the relationship between gaze score and length


valid_length = (
    df[
        [
            "Gaze_Score_Raw",
            "token_count_gaze"
        ]
    ]
    .dropna()
)

pearson_length = pearsonr(
    valid_length[
        "Gaze_Score_Raw"
    ],
    valid_length[
        "token_count_gaze"
    ]
)

spearman_length = spearmanr(
    valid_length[
        "Gaze_Score_Raw"
    ],
    valid_length[
        "token_count_gaze"
    ]
)

print(
    "\nGAZE SCORE VS TOKEN COUNT"
)

print(
    "-------------------------"
)

print(
    f"Pearson r: "
    f"{pearson_length.statistic:.4f}, "
    f"p = {pearson_length.pvalue:.4g}"
)

print(
    f"Spearman rho: "
    f"{spearman_length.statistic:.4f}, "
    f"p = {spearman_length.pvalue:.4g}"
)



# 6 Plot gaze score against passage length


plt.figure(
    figsize=(7, 5)
)

plt.scatter(
    df[
        "token_count_gaze"
    ],
    df[
        "Gaze_Score_Raw"
    ],
    alpha=0.5
)

plt.xlabel(
    "Token Count"
)

plt.ylabel(
    "Raw Gaze Score"
)

plt.title(
    "Raw Gaze Score vs Passage Length"
)

plt.tight_layout()

plt.savefig(
    GAZE_LENGTH_PLOT,
    dpi=300,
    bbox_inches="tight"
)

plt.close()



# 7 Normalise raw gaze scores to create cognitive effort C


c_scaler = MinMaxScaler(
    feature_range=(
        0,
        1
    )
)

df[
    "Cognitive_Effort_C"
] = (
    c_scaler
    .fit_transform(
        df[
            [
                "Gaze_Score_Raw"
            ]
        ]
    )
)

print(
    "\nCognitive-effort signal C:"
)

print(
    df[
        "Cognitive_Effort_C"
    ].describe()
)

print(
    "\nRaw gaze minimum:",
    c_scaler.data_min_[0]
)

print(
    "Raw gaze maximum:",
    c_scaler.data_max_[0]
)



# 8 Confirm the transformation from raw gaze score to C


raw_correlation = pearsonr(
    df[
        "Gaze_Score_Raw"
    ],
    df[
        "Cognitive_Effort_C"
    ]
)

print(
    "\nCorrelation between raw gaze score and C:",
    raw_correlation.statistic
)



# 9 Compare textual difficulty D with cognitive effort C


valid_dc = (
    df[
        [
            "Difficulty_D",
            "Cognitive_Effort_C"
        ]
    ]
    .dropna()
)

pearson_dc = pearsonr(
    valid_dc[
        "Difficulty_D"
    ],
    valid_dc[
        "Cognitive_Effort_C"
    ]
)

spearman_dc = spearmanr(
    valid_dc[
        "Difficulty_D"
    ],
    valid_dc[
        "Cognitive_Effort_C"
    ]
)

print(
    "\nREADABILITY DIFFICULTY D "
    "VS COGNITIVE EFFORT C"
)

print(
    "--------------------------------"
)

print(
    f"Pearson r: "
    f"{pearson_dc.statistic:.4f}, "
    f"p = {pearson_dc.pvalue:.4g}"
)

print(
    f"Spearman rho: "
    f"{spearman_dc.statistic:.4f}, "
    f"p = {spearman_dc.pvalue:.4g}"
)



# 10 Plot the relationship between D and C


plt.figure(
    figsize=(7, 6)
)

plt.scatter(
    df[
        "Difficulty_D"
    ],
    df[
        "Cognitive_Effort_C"
    ],
    alpha=0.5
)

plt.xlabel(
    "Readability Difficulty D"
)

plt.ylabel(
    "Cognitive Effort C"
)

plt.title(
    "Readability Difficulty vs "
    "Predicted Cognitive Effort"
)

plt.grid(
    alpha=0.2
)

plt.tight_layout()

plt.savefig(
    D_C_PLOT,
    dpi=300,
    bbox_inches="tight"
)

plt.close()



# 11 Examine passages where D and C differ most


df[
    "D_C_Difference"
] = (
    df[
        "Difficulty_D"
    ]
    - df[
        "Cognitive_Effort_C"
    ]
)

df[
    "D_C_AbsDifference"
] = (
    df[
        "D_C_Difference"
    ]
    .abs()
)

print(
    "\nD-C difference distribution:"
)

print(
    df[
        "D_C_Difference"
    ].describe()
)

largest_disagreements = (
    df
    .sort_values(
        "D_C_AbsDifference",
        ascending=False
    )
    [
        [
            "Excerpt",
            "Difficulty_D",
            "Cognitive_Effort_C",
            "D_C_Difference"
        ]
    ]
    .head(10)
)

print(
    "\nLargest D-C differences:"
)

print(
    largest_disagreements
)



# 12 Check final Phase 8 data


print(
    "\nMissing D values:",
    df[
        "Difficulty_D"
    ]
    .isna()
    .sum()
)

print(
    "Missing C values:",
    df[
        "Cognitive_Effort_C"
    ]
    .isna()
    .sum()
)

print(
    "Number of passages:",
    len(df)
)



# 13 Export the combined D and C dataset


df.to_csv(
    OUTPUT_PATH,
    index=False
)

print(
    "\nSaved:",
    OUTPUT_PATH
)



# 14 Confirm Phase 8 completion


print(
    "\nPHASE 8 COMPLETE"
)

print(
    "Combined D/C data:",
    OUTPUT_PATH
)

print(
    "Gaze-length plot:",
    GAZE_LENGTH_PLOT
)

print(
    "D-C plot:",
    D_C_PLOT
)

Input file exists: True

Shape: (709, 21)

Input sample:
                                             Excerpt  Difficulty_D  \
0  And outside before the palace a great garden w...      0.635710   
1  One day he had gone beyond any point which he ...      0.443240   
2  So off went Lionel to be made a King. He had n...      0.284857   
3  Father had been away in the country for three ...      0.434188   
4  It was very dark at first, perhaps because the...      0.482255   

   Gaze_Score_Raw  token_count_gaze  
0       -6.495265               223  
1      -26.958176               230  
2      -35.851227               201  
3      -20.772287               220  
4      -21.697758               234  

Raw gaze-score distribution:
count    709.000000
mean     -18.696731
std       14.632622
min      -58.725105
25%      -28.845505
50%      -19.439508
75%       -9.618990
max       33.414940
Name: Gaze_Score_Raw, dtype: float64

Missing gaze scores: 0

Gaze-token count distribution:
count    70